In [ ]:
# === Common notebook preamble (load llm_math package) ===
import sys
from pathlib import Path

# Candidate paths for locating the llm_math package
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# Add parent directories as candidates when running from the notebooks folder
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# Try importing llm_math
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[Warning] failed to load the llm_math package: {e}")
    print("  Clone the GitHub repository and run colab_setup.sh.")
# === End preamble ===


# Ch 07. Backpropagation Math — Mechanical Application of the Chain Rule ⭐

> **Learning Objectives**
> - Derive how the chain rule applies to neural network training
> - Draw computation graphs and compute local derivatives
> - Implement a small autograd engine by hand
> - Experience the CPU vs GPU speed difference in backpropagation

This chapter is the heart of deep learning education. We directly implement what PyTorch's `autograd` does internally.

## 7.1 Why Backpropagation?

To minimize a neural network loss $\mathcal{L}(\theta)$, we need the gradient $\nabla_\theta \mathcal{L}$.

**Forward-mode numerical differentiation**: run one forward pass per parameter.
- $N$ parameters → $N$ forward passes. Slow.

**Reverse-mode automatic differentiation (backpropagation)**: compute all parameter gradients with one backward pass.
- $N$ parameters → 1 forward pass + 1 backward pass. Overwhelmingly faster.


## 7.2 Chain Rule Review

$$\frac{d}{dx} f(g(x)) = f'(g(x)) \cdot g'(x)$$

Extending to multiple variables: when $y = f(u_1, u_2, \ldots, u_n)$ and $u_i = g_i(x)$,

$$\frac{dy}{dx} = \sum_i \frac{\partial f}{\partial u_i} \cdot \frac{du_i}{dx}$$

This is the mathematical core of backpropagation.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Simple composite function: y = f(g(h(x))) = (h(x)^2 + 1) * 2, h(x) = x + 1
# = ((x+1)^2 + 1) * 2
def f(x): return ((x + 1)**2 + 1) * 2
def df_dx(x): return 2 * (x + 1) * 2 * 1  # chain rule

# Verify with numerical differentiation
def numerical_diff(f, x, h=1e-5):
    return (f(x + h) - f(x - h)) / (2 * h)

x0 = 3.0
print(f"f(x) = ((x+1)^2 + 1) * 2")
print(f"f'({x0}) analytic (chain rule): {df_dx(x0)}")
print(f"f'({x0}) numerical: {numerical_diff(f, x0):.6f}")
print(f"Error: {abs(df_dx(x0) - numerical_diff(f, x0)):.2e}")


## 7.3 Computation Graphs

Decompose a complex function into a graph of basic operations such as addition, multiplication, and activations.

Example: $\mathcal{L} = (\sigma(w \cdot x + b) - y)^2$

```
x ──┐
    ├─→ (×) ──→ (+) ──→ (σ) ──→ (-) ──→ (²) ──→ L
w ──┘         ↑              ↑
              b              y
```

Each node only needs to compute its **local gradient**. Backpropagation multiplies these local gradients together through the chain rule.


In [ ]:
# Implement a mini autograd engine
class Tensor:
    """Simple automatic differentiation tensor."""
    def __init__(self, data, requires_grad=False, _children=(), _op=''):
        self.data = np.asarray(data, dtype=float)
        self.requires_grad = requires_grad
        self.grad = None
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __repr__(self):
        return f"Tensor({self.data}, grad={self.grad})"

    @property
    def shape(self):
        return self.data.shape

    # Define operations
    def __add__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(self.data + other.data, _children=(self, other), _op='+')

        def _backward():
            # Local derivative of addition = 1
            self.grad = (self.grad if self.grad is not None else 0) + out.grad
            other.grad = (other.grad if other.grad is not None else 0) + out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(self.data * other.data, _children=(self, other), _op='*')

        def _backward():
            # Local derivatives of multiplication: d/da(a*b) = b, d/db(a*b) = a
            self.grad = (self.grad if self.grad is not None else 0) + other.data * out.grad
            other.grad = (other.grad if other.grad is not None else 0) + self.data * out.grad
        out._backward = _backward
        return out

    def relu(self):
        out = Tensor(np.maximum(0, self.data), _children=(self,), _op='relu')

        def _backward():
            self.grad = (self.grad if self.grad is not None else 0) + (self.data > 0) * out.grad
        out._backward = _backward
        return out

    def sigmoid(self):
        s = 1 / (1 + np.exp(-self.data))
        out = Tensor(s, _children=(self,), _op='sigmoid')

        def _backward():
            self.grad = (self.grad if self.grad is not None else 0) + s * (1 - s) * out.grad
        out._backward = _backward
        return out

    def sum(self):
        out = Tensor(self.data.sum(), _children=(self,), _op='sum')

        def _backward():
            self.grad = (self.grad if self.grad is not None else 0) + np.ones_like(self.data) * out.grad
        out._backward = _backward
        return out

    def backward(self):
        # Topological ordering
        topo = []
        visited = set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build(child)
                topo.append(v)
        build(self)

        self.grad = np.ones_like(self.data)  # dL/dL = 1
        for v in reversed(topo):
            v._backward()

# Test: f(x, y) = (x + y) * (x - y) = x^2 - y^2 (use x*x - y*y for simplicity)
x = Tensor(3.0, requires_grad=True)
y = Tensor(4.0, requires_grad=True)
# z = x*x - y*y  (subtraction = add negative; written directly for implementation simplicity)
a = x * x  # x^2
b = y * y  # y^2
z = a + (b * Tensor(-1.0))  # x^2 - y^2
z.backward()

print(f"x = {x.data}, y = {y.data}")
print(f"z = x^2 - y^2 = {z.data}")
print(f"dz/dx = {x.grad} (Theory 2x = {2*x.data})")
print(f"dz/dy = {y.grad} (Theory -2y = {-2*y.data})")


## 7.4 Deriving Backpropagation for a Linear Layer

Linear layer: $\mathbf{y} = W\mathbf{x} + \mathbf{b}$

Gradients with respect to the loss $\mathcal{L}$:

$$\frac{\partial \mathcal{L}}{\partial W} = \frac{\partial \mathcal{L}}{\partial \mathbf{y}} \cdot \mathbf{x}^\top = \boldsymbol{\delta} \, \mathbf{x}^\top$$

$$\frac{\partial \mathcal{L}}{\partial \mathbf{x}} = W^\top \boldsymbol{\delta}$$

$$\frac{\partial \mathcal{L}}{\partial \mathbf{b}} = \boldsymbol{\delta}$$

Here $\boldsymbol{\delta} = \frac{\partial \mathcal{L}}{\partial \mathbf{y}}$ is the "upstream gradient".


In [ ]:
# Verify linear-layer backpropagation (compare with PyTorch)
import torch

# Input
x_np = np.random.randn(3, 4)  # batch=3, in=4
W_np = np.random.randn(4, 5)  # in=4, out=5
b_np = np.random.randn(5)

# Forward pass + backpropagation in NumPy
def linear_forward(x, W, b):
    return x @ W + b

def linear_backward(x, W, delta):
    """delta: dL/dy (N, out)"""
    dW = x.T @ delta          # (in, out)
    db = delta.sum(axis=0)    # (out,)
    dx = delta @ W.T          # (N, in)
    return dW, db, dx

# Random delta (dL/dy)
delta_np = np.random.randn(3, 5)
y_np = linear_forward(x_np, W_np, b_np)
dW_np, db_np, dx_np = linear_backward(x_np, W_np, delta_np)

# Validate with PyTorch
x_t = torch.tensor(x_np, requires_grad=True)
W_t = torch.tensor(W_np, requires_grad=True)
b_t = torch.tensor(b_np, requires_grad=True)
y_t = x_t @ W_t + b_t
y_t.backward(torch.tensor(delta_np))

print("NumPy vs PyTorch backpropagation comparison:")
print(f"  dW Maximum Error: {np.max(np.abs(dW_np - W_t.grad.numpy())):.2e}")
print(f"  db Maximum Error: {np.max(np.abs(db_np - b_t.grad.numpy())):.2e}")
print(f"  dx Maximum Error: {np.max(np.abs(dx_np - x_t.grad.numpy())):.2e}")
print("\n=> The direct NumPy implementation exactly matches PyTorch autograd!")


## 7.5 Full Derivation of MLP Backpropagation

Two-layer MLP: $\mathbf{o} = W_2 \sigma(W_1 \mathbf{x} + \mathbf{b}_1) + \mathbf{b}_2$

Loss $\mathcal{L} = \|\mathbf{o} - \mathbf{y}\|^2$ (MSE)

Backpropagation:
1. $\frac{\partial \mathcal{L}}{\partial \mathbf{o}} = 2(\mathbf{o} - \mathbf{y})$
2. $\frac{\partial \mathcal{L}}{\partial W_2} = \frac{\partial \mathcal{L}}{\partial \mathbf{o}} \mathbf{h}^\top$
3. $\frac{\partial \mathcal{L}}{\partial \mathbf{h}} = W_2^\top \frac{\partial \mathcal{L}}{\partial \mathbf{o}}$
4. $\frac{\partial \mathcal{L}}{\partial \mathbf{z}_1} = \frac{\partial \mathcal{L}}{\partial \mathbf{h}} \odot \sigma'(\mathbf{z}_1)$
5. $\frac{\partial \mathcal{L}}{\partial W_1} = \frac{\partial \mathcal{L}}{\partial \mathbf{z}_1} \mathbf{x}^\top$

Key idea: repeat **local derivative × upstream gradient** for each layer.


In [ ]:
# Verify MLP backpropagation with numerical differentiation
# f(W1, b1, W2, b2) = ||W2 @ sigmoid(W1 @ x + b1) + b2 - y||^2

def mlp_loss_and_grads(x, W1, b1, W2, b2, y):
    """Forward Pass + Backward Pass (Vector Input, single sample)."""
    # Forward Pass
    z1 = W1 @ x + b1
    h = 1 / (1 + np.exp(-z1))  # sigmoid
    o = W2 @ h + b2
    diff = o - y
    loss = np.sum(diff**2)
    # Backward Pass
    do = 2 * diff
    dW2 = np.outer(do, h)
    db2 = do
    dh = W2.T @ do
    dz1 = dh * h * (1 - h)
    dW1 = np.outer(dz1, x)
    db1 = dz1
    return loss, dW1, db1, dW2, db2

# Numerical differentiation
def numerical_grad(f, param, h=1e-5):
    """Function f returns only the loss. Numerical gradient with respect to param."""
    grad = np.zeros_like(param)
    it = np.nditer(param, flags=['multi_index'])
    while not it.finished:
        idx = it.multi_index
        old = param[idx]
        param[idx] = old + h
        loss_plus = f()
        param[idx] = old - h
        loss_minus = f()
        param[idx] = old
        grad[idx] = (loss_plus - loss_minus) / (2 * h)
        it.iternext()
    return grad

# Validate with a small MLP
np.random.seed(0)
x = np.random.randn(3)
y = np.random.randn(2)
W1 = np.random.randn(4, 3)
b1 = np.random.randn(4)
W2 = np.random.randn(2, 4)
b2 = np.random.randn(2)

loss, dW1, db1, dW2, db2 = mlp_loss_and_grads(x, W1, b1, W2, b2, y)

# Numerical gradient
loss_fn = lambda: mlp_loss_and_grads(x, W1, b1, W2, b2, y)[0]
dW1_num = numerical_grad(loss_fn, W1)
dW2_num = numerical_grad(loss_fn, W2)

print(f"dW1 max error: {np.max(np.abs(dW1 - dW1_num)):.2e}")
print(f"dW2 Maximum Error: {np.max(np.abs(dW2 - dW2_num)):.2e}")
print("=> Analytic backward pass and numerical differentiation match!")


## 7.6 Vanishing and Exploding Gradients

The sigmoid derivative satisfies $\sigma'(x) = \sigma(x)(1-\sigma(x)) \leq 0.25$.

In deep neural networks, repeatedly multiplying these small values during backpropagation makes gradients shrink exponentially → **vanishing gradients**.

Common remedies:
- ReLU-family activations (derivatives are 0 or 1)
- Proper weight initialization (He, Xavier)
- Residual connections (introduced later in Transformers)
- BatchNorm/LayerNorm


In [ ]:
# Simulate vanishing gradients
def sigmoid(x): return 1 / (1 + np.exp(-x))
def sigmoid_deriv(x):
    s = sigmoid(x)
    return s * (1 - s)

# Gradient magnitude by depth
np.random.seed(0)
depths = [5, 10, 20, 50, 100]
print(f"{'Depth':>6} | {'Mean |grad|':>15} | {'Minimum |grad|':>15}")
print("-" * 45)
for d in depths:
    x = np.random.randn(100)
    grad = np.ones(100)  # dL/dy
    for _ in range(d):
        # Backward Pass sigmoid layers
        z = np.random.randn(100)
        grad = grad * sigmoid_deriv(z)
    print(f"{d:>6} | {np.mean(np.abs(grad)):>15.2e} | {np.min(np.abs(grad)):>15.2e}")

print("\n=> As depth increases, gradients shrink exponentially (vanishing gradients)!")
print("With ReLU, derivatives are 0 or 1, so this problem is greatly reduced.")


## 7.7 [CPU/GPU Benchmark ②] Backpropagation Time Comparison

Compare forward+backward time on CPU vs GPU for different model sizes.


In [ ]:
# Backpropagation benchmark
import torch
import time
from llm_math.bench import time_fn, format_results_table

def make_mlp_and_run(input_dim, hidden, output_dim, n_layers, batch_size=32, device='cpu'):
    """Build an MLP and run one forward+backward step."""
    dims = [input_dim] + [hidden] * (n_layers - 1) + [output_dim]
    layers = [torch.nn.Linear(dims[i], dims[i+1]) for i in range(len(dims) - 1)]
    model = torch.nn.Sequential(*layers).to(device)
    x = torch.randn(batch_size, input_dim, device=device)
    y = torch.randn(batch_size, output_dim, device=device)
    loss_fn = torch.nn.MSELoss()
    opt = torch.optim.SGD(model.parameters(), lr=0.01)

    def step():
        opt.zero_grad()
        out = model(x)
        loss = loss_fn(out, y)
        loss.backward()
        opt.step()
        return loss
    return step

# Compare by model size
configs = [
    ('small',  100, 64, 10, 3),
    ('medium', 100, 256, 10, 5),
    ('large',  100, 512, 10, 10),
    ('xlarge', 100, 1024, 10, 20),
]

print(f"{'Config':>8} | {'CPU (ms)':>12} | {'GPU (ms)':>12} | {'Speedup':>10}")
print("-" * 50)
for name, in_d, hid, out_d, nl in configs:
    step_cpu = make_mlp_and_run(in_d, hid, out_d, nl, device='cpu')
    res_cpu = time_fn(step_cpu, device='cpu', warmup=2, repeat=5)
    if torch.cuda.is_available():
        step_gpu = make_mlp_and_run(in_d, hid, out_d, nl, device='cuda')
        res_gpu = time_fn(step_gpu, device='cuda', warmup=3, repeat=5)
        speedup = res_cpu['mean_ms'] / res_gpu['mean_ms']
        print(f"{name:>8} | {res_cpu['mean_ms']:>12.3f} | {res_gpu['mean_ms']:>12.3f} | {speedup:>9.2f}x")
    else:
        print(f"{name:>8} | {res_cpu['mean_ms']:>12.3f} | {'N/A':>12} | {'N/A':>10}")

if not torch.cuda.is_available():
    print("\n⚠ No GPU detected. Switch to a T4 GPU runtime in Colab to see acceleration.")
    print("  Especially for the large model (xlarge), CPU takes seconds while GPU takes tens of ms.")


## 7.8 Key Takeaways

| Concept | Formula | Meaning |
|---|---|---|
| Chain rule | $\frac{d}{dx}f(g(x)) = f'(g)g'(x)$ | Differentiate composite functions |
| Computation graph | — | Decompose a complex function into basic operations |
| Local gradient | — | Derivative of a node's output with respect to its input |
| Backpropagation | $\nabla_\theta \mathcal{L}$ | All gradients from one backward pass |
| Linear-layer backprop | $\partial\mathcal{L}/\partial W = \delta \mathbf{x}^\top$ | Weight gradient |

## Exercises

1. Add `__sub__`, `__truediv__`, and `matmul` operations to the mini autograd `Tensor` class.
2. Use mini autograd to compute $\partial f/\partial x$ and $\partial f/\partial y$ for $f(x, y) = (x + y)^2 - xy$, and verify them with numerical differentiation.
3. Build a 3-layer MLP in PyTorch and print `W1.grad` and `W2.grad` after one backward step.
4. Compare gradient norms in MLPs with depths 1, 5, 10, and 20 to demonstrate vanishing gradients.
5. Compare He initialization with a poor initialization (plain `randn`) and observe how deep MLP training changes.

> Solutions: `solutions/ch07_solutions.ipynb`
